# InsuranceQA Gemma 4 — evaluate a finished fine-tune (short notebook)

Loads the **already trained adapter** on the Gemma 4 E4B base and runs everything after training:
validation generation and ranking, export, fresh-process reload check, final test generation and metrics.
**No training is repeated.** It restores your earlier run folder, detects that all epochs finished and goes straight to evaluation.

**Before you run (Save Version → Save & Run All):**
1. Settings: Accelerator **GPU T4 x2**, **Internet on**.
2. Add-ons → Secrets: `HF_TOKEN` (write access) attached and switched on. Your previous run backed up to a private repo named after the run folder.
3. Edit the settings in the first code cell (old run name and restore source), nothing else is required.


In [1]:
import os, sys, json, time, subprocess, shutil, base64, io, zipfile
from pathlib import Path
assert sys.platform.startswith("linux"), "Run in Kaggle's Linux GPU environment."
assert (3, 11) <= sys.version_info[:2] < (3, 14), "Use Python 3.11-3.13."

# ============================ YOUR SETTINGS ============================
OLD_RUN_NAME   = "insuranceqa_20261031_194809"   # folder name of the finished training run
RESTORE_FROM   = "hf"      # "hf" = pull from your private Hugging Face backup repo (needs HF_TOKEN)
                           # or a path to the old run folder attached as notebook output, e.g.
                           # "/kaggle/input/<your-notebook>/insuranceqa_20261031_194809"
HOURS          = 8.0       # work budget for this session (Kaggle cap is 12h; install/download/backup add ~0.7h)
SELECTION_N    = 64        # samples per candidate when ranking checkpoints (raise if you have spare time)
FINAL_N        = 128       # samples in the final test generation
TEMPERATURES   = [0.9, 0.7, 1.0]
EVALUATE_BASE  = True      # also score the untuned base model as a control (done last, first to be cut if time is short)
BACKUP_REQUIRED = True     # push results to your HF repo during/after the run
BACKUP_EVERY_MIN = 30
# =======================================================================

RESUME_RUN_NAME, RESUME_SOURCE = OLD_RUN_NAME, RESTORE_FROM
RUN_NAME = OLD_RUN_NAME            # same name => absolute checkpoint paths inside the restored files stay valid
RUN_DIR = Path("/kaggle/working") / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
PROGRAM = Path("/kaggle/working/insurance_gemma_program"); (PROGRAM / "src").mkdir(parents=True, exist_ok=True)
CONFIG_PATH = RUN_DIR / "config.json"   # restored from the old run in the run cell; not written here
OVERRIDE_KEYS = ["hours", "selection_n", "final_n", "selection_temperatures", "evaluate_base"]
CONFIG = {"hours": HOURS, "selection_n": SELECTION_N, "final_n": FINAL_N, "selection_temperatures": TEMPERATURES,
          "evaluate_base": EVALUATE_BASE, "dataset": "insuranceqa", "run_dir": str(RUN_DIR)}
os.environ["HF_HOME"] = "/tmp/insurance_gemma_hf"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
HF_TOKEN_PRESENT = False
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN"); HF_TOKEN_PRESENT = True
except Exception:
    pass
print("Run folder:", RUN_DIR, "| HF_TOKEN attached:", HF_TOKEN_PRESENT)


Run folder: /kaggle/working/insuranceqa_20261031_194809 | HF_TOKEN attached: True


In [2]:
# Program files packaged from the full notebook (core, metrics, model_runtime, prepare, run, train, worker, backup + requirements).
PAYLOAD = """
...
"""
zipfile.ZipFile(io.BytesIO(base64.b64decode("".join(PAYLOAD.split())))).extractall(PROGRAM)
print(sorted(p.name for p in (PROGRAM / "src").glob("*.py")), (PROGRAM / "requirements.txt").exists())


['backup.py', 'core.py', 'metrics.py', 'model_runtime.py', 'prepare.py', 'run.py', 'train.py', 'worker.py'] True


In [3]:
gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"], text=True)
print(gpu_info)
assert len(gpu_info.strip().splitlines()) == 2, "Select GPU T4 x2 in Kaggle settings."
# Kaggle sessions are disposable. Using the active interpreter avoids the Python 3.13
# ensurepip/venv failure and lets child processes use the packages just installed.
PY = Path(sys.executable)
install_log = RUN_DIR / "installation.log"
commands = [
    [str(PY), "-m", "pip", "install", "--upgrade", "pip"],
    [str(PY), "-m", "pip", "install", "--upgrade-strategy", "only-if-needed", "-r", str(PROGRAM / "requirements.txt")],
]
with install_log.open("w") as log:
    for cmd in commands:
        print("Installing/checking:", " ".join(cmd[3:]), flush=True)
        result = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT)
        log.flush()
        if result.returncode:
            print(install_log.read_text()[-8000:])
            raise RuntimeError(f"Environment setup failed. Full log: {install_log}")
# `pip check` audits every unrelated package in Kaggle's shared image. Its existing
# conflicts (for example, a Colab-only package) do not tell us whether this pipeline works.
# Check exactly the runtime libraries and CUDA availability used by this notebook instead.
health = """
import torch, transformers, trl, peft, datasets, sentence_transformers, bitsandbytes, unsloth
print('torch', torch.__version__, 'cuda', torch.version.cuda)
print('transformers', transformers.__version__)
print('trl', trl.__version__)
print('peft', peft.__version__)
print('datasets', datasets.__version__)
print('cuda_available', torch.cuda.is_available())
print('cuda_device_count', torch.cuda.device_count())
assert torch.cuda.is_available() and torch.cuda.device_count() == 2
"""
probe = subprocess.check_output([str(PY), "-c", health], text=True, stderr=subprocess.STDOUT)
print(probe)
print("Environment installed. Full compatibility and numerical checks run next.")


Tesla T4, 580.178.04, 15360 MiB
Tesla T4, 580.178.04, 15360 MiB

Installing/checking: install --upgrade pip
Installing/checking: install --upgrade-strategy only-if-needed -r /kaggle/working/insurance_gemma_program/requirements.txt
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1666: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._

In [4]:
import signal
PREPARE_ONLY = False
BACKUP_SCRIPT = str(PROGRAM / "src/backup.py")

def backup_cmd(cmd):
    args = [str(PY), BACKUP_SCRIPT, cmd, "--run-name", RUN_NAME]
    if cmd != "check": args += ["--run-dir", str(RUN_DIR)]
    return args

def run_backup_blocking(cmd, timeout=900):
    try:
        out = subprocess.run(backup_cmd(cmd), capture_output=True, text=True, timeout=timeout)
        line = (out.stdout.strip().splitlines() or ["{}"])[-1]
        return json.loads(line)
    except Exception as exc:
        return {"ok": False, "error": f"{type(exc).__name__}: {exc}"}

# --- 1. backup must work BEFORE spending GPU time
backup_ok = False
if HF_TOKEN_PRESENT:
    res = run_backup_blocking("check", 300); backup_ok = bool(res.get("ok"))
    print("Backup check:", res)
if not backup_ok:
    msg = ("Off-Kaggle backup is NOT working (add a Kaggle secret HF_TOKEN with WRITE access and attach it to this notebook). "
           "Kaggle deletes /kaggle/working if a session is killed or the quota runs out.")
    if BACKUP_REQUIRED: raise RuntimeError(msg + " Set BACKUP_REQUIRED=False to run without protection.")
    print("WARNING:", msg)

# --- 2. optional resume of an interrupted run
if RESUME_RUN_NAME:
    assert RESUME_SOURCE, "Set RESUME_SOURCE to 'hf' or a path to the old run folder."
    if RESUME_SOURCE == "hf":
        res = run_backup_blocking("pull"); print("Restore:", res); assert res.get("ok"), "Restore from Hugging Face failed."
    else:
        shutil.copytree(RESUME_SOURCE, RUN_DIR, dirs_exist_ok=True)
    old = json.loads(CONFIG_PATH.read_text())
    old.update({k: CONFIG[k] for k in OVERRIDE_KEYS}, resume=True, run_dir=str(RUN_DIR)); old.pop("model_path", None)
    CONFIG.update(old); CONFIG_PATH.write_text(json.dumps(old, indent=2))
    print("Resuming run", RUN_NAME)

# --- 3. run the pipeline in the background and supervise it
cmd = [str(PY), "-u", str(PROGRAM / "src/run.py"), "--config", str(CONFIG_PATH)]
if PREPARE_ONLY:
    cmd.append("--prepare-only")
pipeline_log = RUN_DIR / "pipeline.log"
wall_limit = CONFIG["hours"] * 3600 + 900
began = time.monotonic(); last_beat = began; last_backup = began; backup_proc = None; pipeline_error = None
def tail_line(path):
    try:
        text = Path(path).read_bytes()[-4000:].decode("utf-8", "replace").replace("\r", "\n")
        lines = [l for l in text.splitlines() if l.strip()]
        return lines[-1][:300] if lines else ""
    except Exception: return ""
def start_backup():
    global backup_proc, last_backup
    if backup_ok and (backup_proc is None or backup_proc.poll() is not None):
        backup_proc = subprocess.Popen(backup_cmd("push"), stdout=open(RUN_DIR / "backup.log", "a"), stderr=subprocess.STDOUT)
        last_backup = time.monotonic()
try:
    with pipeline_log.open("w") as log:
        process = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        previous = ""
        while process.poll() is None:
            time.sleep(15)
            status_file = RUN_DIR / "session_status.json"
            current = status_file.read_text() if status_file.exists() else "Starting..."
            if current != previous:
                print(current, flush=True); previous = current
            now = time.monotonic()
            if now - last_beat > 600:   # heartbeat: last training/generation log line every 10 minutes
                last_beat = now
                print(f"[{(now-began)/3600:.2f}h] " + tail_line(RUN_DIR / "logs/train.log"), flush=True)
            if now - last_backup > BACKUP_EVERY_MIN * 60: start_backup()
            if now - began > wall_limit:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=180)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL)
                raise TimeoutError("Notebook watchdog stopped the controller. Logs and any exported adapter are preserved.")
        if process.returncode:
            pipeline_error = f"Pipeline stopped with exit code {process.returncode}. Inspect {RUN_DIR / 'failure_report.json'} and the logs."
except KeyboardInterrupt:
    if 'process' in globals() and process.poll() is None:
        os.killpg(process.pid, signal.SIGTERM)
    raise
except Exception as exc:
    pipeline_error = f"{type(exc).__name__}: {exc}"

# --- 4. final backup, then report. Deliberately does NOT raise: the notebook must finish normally so Kaggle commits /kaggle/working.
if backup_proc is not None and backup_proc.poll() is None:
    try: backup_proc.wait(timeout=900)
    except Exception: pass
if backup_ok:
    print("Final backup:", run_backup_blocking("push", 1800))
print(pipeline_log.read_text()[-8000:])
if pipeline_error:
    print("\n" + "!" * 78 + "\nPIPELINE DID NOT COMPLETE:", pipeline_error,
          "\nAny adapter that was saved is in adapter_DIAGNOSTIC_ONLY.zip / adapter_package_preliminary.zip.\n" + "!" * 78)


Backup check: {'ok': True, 'cmd': 'check', 'repo': 'adeelumar17/insuranceqa_20261031_194809', 'time': '12:02:50'}
Restore: {'ok': True, 'cmd': 'pull', 'repo': 'adeelumar17/insuranceqa_20261031_194809', 'time': '12:03:08'}
Resuming run insuranceqa_20261031_194809
{
  "status": "running",
  "stage": "tokenization_and_context_check",
  "elapsed_hours": 0.0008665111329820421,
  "resume": true
}
{
  "status": "running",
  "stage": "download_exact_base_weights",
  "elapsed_hours": 0.02003940317365858,
  "resume": true
}
{
  "status": "running",
  "stage": "validation_generation",
  "elapsed_hours": 0.043429893917507595,
  "resume": true
}
[0.17h] [rank0]:[W1101 04:23:33.035993562 ProcessGroupNCCL.cpp:1575] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())
[0.33h] [rank0]:[W1101 04:23:33.035993562 ProcessGroupNCCL.cpp:1575] Warning: WA

In [5]:
import zipfile
from IPython.display import display, FileLink
def load(name):
    p = RUN_DIR / name
    return json.loads(p.read_text()) if p.exists() else None
status, sel, fin, trn = load("session_status.json"), load("selection_report.json"), load("final_report.json"), load("training_report.json")
print("STATUS:", json.dumps({k: status.get(k) for k in ("status", "validation_acceptance_passed", "stage_errors", "elapsed_hours", "error")} if status else {}, indent=2))
if trn: print("training (from earlier session): steps", trn.get("steps"), "epoch", trn.get("epoch"), "stop", trn.get("stop_reason"))
if sel:
    c = sel["chosen"]
    print("chosen:", c.get("name"), "| fallback (unranked):", c.get("diagnostic_only", False), "| val_loss:", c.get("validation_loss"), "| normalized_score:", c.get("normalized_score"))
    print("partial candidates:", sel.get("partial_candidates"), "| not scored:", sel.get("incomplete_candidates"))
    for r in sel.get("candidates", []) + sel.get("baseline", []):
        print("  ", r.get("name"), "| JSON validity", r.get("raw_schema_validity"), "| score", r.get("normalized_score"), "| partial", r.get("partial"))
if fin:
    print("final generation complete:", fin.get("complete"), fin.get("generation_status"))
    print("raw JSON validity:", fin["raw"].get("raw_schema_validity"), "| accepted records:", fin["accepted"].get("metric_records"), "| rejection rate:", fin.get("rejection_rate"))

# One bundle with everything worth keeping (no optimizer checkpoints, raw downloads or token files).
bundle = Path("/kaggle/working/evaluation_results_bundle.zip")
skip_dirs = ("checkpoints", "raw", "tokens", "smoke", "smoke_adapter", "tokenizer")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for p in RUN_DIR.rglob("*"):
        rel = p.relative_to(RUN_DIR)
        if not p.is_file() or rel.parts[0] in skip_dirs or rel.parts[0].startswith("export_superseded") or p.suffix == ".zip": continue
        z.write(p, str(rel))
    for name in ("adapter_package.zip", "adapter_package_preliminary.zip", "final_training_adapter.zip", "adapter_DIAGNOSTIC_ONLY.zip"):
        if (RUN_DIR / name).exists(): z.write(RUN_DIR / name, name)
print("\nBundle:", bundle, round(bundle.stat().st_size / 1e6, 1), "MB")
for p in [bundle] + [RUN_DIR / n for n in ("adapter_package.zip", "final/synthetic.jsonl", "final/manual_review.jsonl", "final_report.json", "selection_report.json")]:
    if p.exists(): display(FileLink(str(p)))


STATUS: {
  "status": "partial",
  "validation_acceptance_passed": false,
  "stage_errors": [],
  "elapsed_hours": 7.348328010837237,
  "error": null
}
training (from earlier session): steps 2412 epoch 3.0 stop epoch_limit
chosen: adapter_0_t0.9 | fallback (unranked): False | val_loss: 3.085557222366333 | normalized_score: 0.9422123544149867
partial candidates: ['adapter_0_t0.9'] | not scored: ['adapter_0_t0.7', 'adapter_0_t1.0']
   adapter_0_t0.9 | JSON validity 0.022222222222222223 | score 0.9422123544149867 | partial True
   base_t0.9 | JSON validity 1.0 | score 2.0067534235879596 | partial False
   base_t0.7 | JSON validity 1.0 | score 2.40806660367003 | partial False
   base_t1.0 | JSON validity 1.0 | score 1.9360233554365989 | partial False
final generation complete: False {'raw': {'requested': 128, 'requests_processed': 47, 'attempts': 47, 'accepted': 0, 'returned': 0, 'shortfall': 128, 'rejections': {'not_object': 47, 'length_limited': 9, 'deadline_hit': 1}, 'complete': False, 

/kaggle/working/evaluation_results_bundle.zip

/kaggle/working/insuranceqa_20261031_194809/adapter_package.zip

/kaggle/working/insuranceqa_20261031_194809/final/synthetic.jsonl

/kaggle/working/insuranceqa_20261031_194809/final/manual_review.jsonl

/kaggle/working/insuranceqa_20261031_194809/final_report.json

/kaggle/working/insuranceqa_20261031_194809/selection_report.json